# 3. Translators

The translator is Ministral 3 8B in its base version (`mistralai/Ministral-3-8B-Base-2512`, Apache 2.0), adapted with
QLoRA: the base stays frozen in 4-bit NF4 and rank 16 adapters are trained on the seven linear projections of every
block, about 0.55 % of the parameters. Each adapter is trained from the base model.

| Translator | Mixture | Steps | Max tokens |
|---|---|---|---|
| v2 | FOLIO training stories and DGT sentences formalized by the base model | 300 | 768 |
| v3 | adds synthetic stories, ContractNLI and renamed FOLIO stories | 400 | 1024 |
| v4 | adds stories over real vocabularies in six languages | 500 | 1024 |

Needs a GPU with 24 GB (the paper used one NVIDIA L4), about 40 GB of disk, and `pip install -e ".[gpu]"`. Each
translator takes about an hour. Run notebook 2 first.

In [ ]:
# Copyright 2026 Javier Marín
# SPDX-License-Identifier: Apache-2.0

from pathlib import Path

REPO = Path.cwd().parent
DATA = REPO / "data"
RUNS = REPO / "runs"      # everything this notebook produces goes here; data/ is never overwritten

from logicverifier import data, models

TRAINING = DATA / "training"
BASE = models.download_base_model(RUNS / "models" / "base")

## Translator v2

The base model writes each English DGT sentence in logic. A formula is kept when it parses, invents no number, keeps
every quantity and negates exactly when the sentence does; it then becomes the answer for the same sentence in all six
languages.

In [ ]:
llm = models.load_llm(BASE)
dgt_train = data.read_jsonl(TRAINING / "dgt_train.jsonl")
dgt_formulas = models.formalize_dgt(llm, dgt_train)
del llm   # restart the runtime here if the GPU memory is not released

folio_train = data.read_jsonl(RUNS / "folio" / "folio-train.jsonl")
mixture_v2, report_v2 = data.build_v2_mixture(folio_train, dgt_train, dgt_formulas)
data.write_jsonl(mixture_v2, RUNS / "training" / "mixture_v2.jsonl")
report_v2

In [ ]:
models.train_adapter(BASE, RUNS / "training" / "mixture_v2.jsonl", RUNS / "models" / "v2_adapter", steps=300, max_len=768)

## ContractNLI formalized by translator v2

Translator v2 writes each case with its own names, greedily and three more times at temperature 0.7. The first
translation that Z3 decides with the human label is kept.

In [ ]:
merged_v2 = models.merge_adapter(BASE, RUNS / "models" / "v2_adapter", RUNS / "models" / "v2_merged")
llm = models.load_llm(merged_v2)
contract_verified, contract_report = models.contract_pseudo_labels(llm, data.read_jsonl(TRAINING / "contractnli_cases.jsonl"))
data.write_jsonl(contract_verified, RUNS / "training" / "contractnli_verified.jsonl")
contract_report

## Translators v3 and v4

The mixtures come from notebook 2.

In [ ]:
models.train_adapter(BASE, RUNS / "training" / "mixture_v3.jsonl", RUNS / "models" / "v3_adapter", steps=400, max_len=1024)

In [ ]:
models.train_adapter(BASE, RUNS / "training" / "mixture_v4.jsonl", RUNS / "models" / "v4_adapter", steps=500, max_len=1024)